# PE_V2X_Reliability — Guide de reproductibilité & guide technique (Schéma A)

Ce guide fournit des étapes de reproduction adaptées à VS Code, le contrat run_id, les limites de modification de paramètres, et une documentation fichier par fichier (scripts + modules).

**Date:** 2026-03-01

## 1 Reproduction (terminal intégré VS Code ; indépendant du chemin)

Ouvrir la racine du dépôt dans VS Code (notée `<ROOT>`), puis *Terminal → New Terminal*. Sur Windows, vous pouvez choisir **Command Prompt / PowerShell** ; les deux écritures sont proposées.

### 1.1 Créer et activer un environnement virtuel (Windows)

**Option A : Command Prompt**

```bat
cd <ROOT>\03_sim_A\py
python -m venv <ROOT>\.venv
<ROOT>\.venv\Scripts\activate.bat
pip install -r <ROOT>\06_report\requirements_utf8.txt
```

**Option B : PowerShell**

```powershell
cd <ROOT>\03_sim_A\py
python -m venv <ROOT>\.venv
<ROOT>\.venv\Scripts\Activate.ps1
pip install -r <ROOT>\06_report\requirements_utf8.txt
```

### 1.2 Smoke (valider le contrat de sortie)

```text
python run_pipeline_A.py --run_id A_Smoke --preset RefPlus --scenarios Ref --rets 0,1,2 --seed_start 1 --n_seeds 1 --duration_s 60 --msg_rate_hz 10 --tx_mode mix --tx_k 6 --tx_k_cross 2
```

### 1.3 small / S20

- **small** : `--n_seeds` = 3–5 pour vérifier les tendances (Fig03/04/05/06/07).  
- **S20** : utiliser `run_commands.txt` comme snapshot protocolaire de référence.


## 2 Contrat de sortie run_id

Chaque run écrit dans `<ROOT>\05_results_A\runs\<run_id>\` : `raw/`, `tables/`, `figures/`, `run_commands.txt`.

Politique “review-first” : conserver `tables/` + `run_commands.txt`. Si stockage limité, supprimer raw après agrégation.

## 3 Limites de modification des paramètres

- **Sûr** : options CLI (rets, msg_rate, deadline, seeds, switch congestion).
- **Moyen** : presets scénario et modules de propagation.
- **Risqué** : `mac_congestion.py` (mécanisme Cong-only) ; valider Fig03/Fig04 après modification.

## 4 Documentation fichier par fichier



### 4.1 Scripts principaux (`03_sim_A/py`)

### analyze_metrics_A.py

**Rôle :** Agrégation : raw → tables (CSV lisibles).
- Taille : ~**335** lignes
- Arguments CLI (extrait) : `--band_max_m`, `--band_min_m`, `--dist_bin_m`, `--mid_bin_m`, `--min_success_per_bin`, `--min_total_per_bin`, `--retrans`, `--run_id`, `--scenario`, `--u_bin_w`, `--u_max`, `--u_min`

**Dépendances principales (extrait) :**
`__future__.annotations`, `argparse`, `pathlib.Path`, `numpy`, `pandas`, `paths_A.ensure_run_dirs_a`, `paths_A.make_run_id`, `paths_A.load_latest_run_id`, `run_logging.log_command`, `run_logging.update_manifest`

**Structures clés :**
- Fonctions :
  - `_pick_run_id(arg_run_id)`
  - `_quantiles_ms(x)`
  - `_pick_latest_packets_file(raw_dir, scenario, ret)`
  - `main()`

**Artéfacts potentiels (scan de chaînes, extrait) :**
- `position_heterogeneity__UrbMask__ret{args.retrans}__band{int(band_min)}-{int(band_max)}__{tag}.csv`
- `results_packets__{scenario}__ret{ret}.csv`
- `results_packets__{scenario}__ret{ret}__seed*.csv`
- `summary_metrics__{args.scenario}__ret{args.retrans}__{tag}.csv`
- `tunnel_segments__Tunnel__ret{args.retrans}__band{int(band_min)}-{int(band_max)}__{tag}.csv`

**Paramètres recommandés (un à la fois, puis small seeds) :**
- Utiliser plutôt preset/CLI ; si constants internes modifiées, documenter dans le rapport et snapshots.

### generate_trajectories_A.py

**Rôle :** Génération : trajectoires (géométrie + IDM + feux + cross/turn).
- Taille : ~**649** lignes
- Arguments CLI (extrait) : `--cross_enable`, `--cross_half_length_m`, `--dt_s`, `--duration_s`, `--flow_cross_vph`, `--flow_main_vph`, `--i1_x`, `--i2_x`, `--idm_T_s`, `--idm_a_mps2`, `--idm_b_mps2`, `--idm_delta` …

**Dépendances principales (extrait) :**
`__future__.annotations`, `argparse`, `numpy`, `pandas`, `paths_A.ensure_run_dirs_a`, `paths_A.make_run_id`, `paths_A.load_latest_run_id`, `run_logging.log_command`, `run_logging.update_manifest`, `run_logging.snapshot_file`, `modules.road_geometry.RefPlusGeometry`, `modules.traffic_signals.SignalPlan`, `modules.traffic_idm.IDMParams`, `modules.traffic_idm.idm_accel`

**Structures clés :**
- Fonctions :
  - `_pick_run_id(arg_run_id)`
  - `_ensure_time_key(df)`
  - `_spawn_ok(vlist, min_spawn_gap_m)`
  - `_road_tag_main(direction, lane_id)`
  - `_road_tag_cross(which, direction, lane_id)`
  - `_road_tag_turn(which, turn_kind, lane_id)`
  - `_lane_key(kind, which, direction, lane_id)`
  - `_init_lane_state(geom, n_lanes_per_dir, cross_enable)`
  - `_simulate_refplus_idm(geom, plan_i1, plan_i2, flow_main_vph, flow_cross_vph, p_turn_i1, p_turn_i2, p_right, p_left, veh_length_m…)`
  - `main()`

**Artéfacts potentiels (scan de chaînes, extrait) :**
- `traj__Ref.csv`

**Paramètres recommandés (un à la fois, puis small seeds) :**
- Utiliser plutôt preset/CLI ; si constants internes modifiées, documenter dans le rapport et snapshots.

### generate_tunnel_config_A.py

**Rôle :** Génération : config tunnel (intervalle + transition + dégradation + queue).
- Taille : ~**77** lignes
- Arguments CLI (extrait) : `--b_floor`, `--b_peak`, `--bell_gamma`, `--delay_exp_scale_ms`, `--delay_extra_ms`, `--run_id`, `--scenario`, `--severity`, `--transition_m`, `--x0_m`, `--x1_m`

**Dépendances principales (extrait) :**
`__future__.annotations`, `argparse`, `pandas`, `paths_A.ensure_run_dirs_a`, `paths_A.make_run_id`, `paths_A.load_latest_run_id`, `run_logging.log_command`, `run_logging.update_manifest`, `run_logging.snapshot_file`, `modules.prop_tunnel.TunnelConfig`

**Structures clés :**
- Fonctions :
  - `_pick_run_id(arg_run_id)`
  - `main()`

**Artéfacts potentiels (scan de chaînes, extrait) :**
- `tunnel_config__{args.scenario}.csv`

**Paramètres recommandés (un à la fois, puis small seeds) :**
- Utiliser plutôt preset/CLI ; si constants internes modifiées, documenter dans le rapport et snapshots.

### generate_urbmask_buildings_A.py

**Rôle :** Génération : bâtiments UrbMask (rectangles + hauteurs).
- Taille : ~**83** lignes
- Arguments CLI (extrait) : `--max_h_m`, `--max_height_m`, `--max_w_m`, `--min_h_m`, `--min_height_m`, `--min_w_m`, `--n_blocks`, `--road_length_m`, `--run_id`, `--scenario`, `--seed`, `--x_margin_m` …

**Dépendances principales (extrait) :**
`__future__.annotations`, `argparse`, `paths_A.ensure_run_dirs_a`, `paths_A.make_run_id`, `paths_A.load_latest_run_id`, `run_logging.log_command`, `run_logging.update_manifest`, `run_logging.snapshot_file`, `modules.buildings_3d.generate_buildings`, `modules.buildings_3d.save_buildings_csv`

**Structures clés :**
- Fonctions :
  - `_pick_run_id(arg_run_id)`
  - `main()`

**Artéfacts potentiels (scan de chaînes, extrait) :**
- `buildings__{args.scenario}__seed{args.seed}.csv`

**Paramètres recommandés (un à la fois, puis small seeds) :**
- Utiliser plutôt preset/CLI ; si constants internes modifiées, documenter dans le rapport et snapshots.

### paths_A.py

**Rôle :** Conventions de chemins / nommage.
- Taille : ~**156** lignes

**Dépendances principales (extrait) :**
`__future__.annotations`, `dataclasses.dataclass`, `pathlib.Path`, `datetime.datetime`, `json`

**Structures clés :**
- Classes :
  - `BasePathsA`
  - `RunPathsA`
- Fonctions :
  - `_detect_project_root(from_file)`
  - `get_base_paths_a()`
  - `make_run_id(prefix)`
  - `ensure_base_dirs_a()`
  - `ensure_run_dirs_a(run_id, save_as_latest, meta)`
  - `load_latest_run_id()`

**Artéfacts potentiels (scan de chaînes, extrait) :**
- `LATEST_RUN.json`
- `run_manifest.json`

**Paramètres recommandés (un à la fois, puis small seeds) :**
- Utiliser plutôt preset/CLI ; si constants internes modifiées, documenter dans le rapport et snapshots.

### plot_figures_A.py

**Rôle :** Tracés deliverable : F1/F2/F3 depuis tables.
- Taille : ~**264** lignes
- Arguments CLI (extrait) : `--cdf_max_dist_m`, `--curve_style`, `--min_bin_count`, `--retrans`, `--run_id`, `--scenario`, `--smooth_overlay_raw`, `--smooth_window_m`, `--x_max_m`

**Dépendances principales (extrait) :**
`__future__.annotations`, `argparse`, `pathlib.Path`, `numpy`, `pandas`, `matplotlib.pyplot`, `paths_A.ensure_run_dirs_a`, `paths_A.make_run_id`, `paths_A.load_latest_run_id`, `run_logging.log_command`, `run_logging.update_manifest`

**Structures clés :**
- Fonctions :
  - `_pick_run_id(arg_run_id)`
  - `ecdf(x)`
  - `_pick_latest_summary_file(tables_dir, scenario, ret)`
  - `_pick_latest_packets_file(raw_dir, scenario, ret)`
  - `_smooth_by_distance(x, y, window_m)`
  - `main()`

**Artéfacts potentiels (scan de chaînes, extrait) :**
- `F1_PDR_vs_distance__{scenario}__ret{ret}__{tag}.png`
- `F2_Delay_CDF__{scenario}__ret{ret}__{tag_f2}.png`
- `F3_Delay_p95_p99_vs_distance__{scenario}__ret{ret}__{tag}.png`
- `results_packets__{scenario}__ret{ret}.csv`
- `summary_metrics__{scenario}__ret{ret}__*.csv`

**Paramètres recommandés (un à la fois, puis small seeds) :**
- Utiliser plutôt preset/CLI ; si constants internes modifiées, documenter dans le rapport et snapshots.

### progress_util.py

**Rôle :** Utilitaires de progression.
- Taille : ~**43** lignes

**Dépendances principales (extrait) :**
`__future__.annotations`, `typing.Iterable`, `typing.Iterator`, `typing.Optional`, `typing.TypeVar`, `sys`

**Structures clés :**
- Fonctions :
  - `progress(it, total, desc)`

**Paramètres recommandés (un à la fois, puis small seeds) :**
- Utiliser plutôt preset/CLI ; si constants internes modifiées, documenter dans le rapport et snapshots.

### run_logging.py

**Rôle :** Journalisation d’audit.
- Taille : ~**97** lignes

**Dépendances principales (extrait) :**
`__future__.annotations`, `json`, `os`, `shutil`, `sys`, `datetime.datetime`, `pathlib.Path`, `typing.Any`, `typing.Dict`, `typing.Optional`

**Structures clés :**
- Fonctions :
  - `_now()`
  - `log_command(run_id, run_results_dir, extra)`
  - `update_manifest(manifest_path, patch)`
  - `snapshot_file(src, run_results_dir, category, rename_to)`
  - `_quote(s)`

**Artéfacts potentiels (scan de chaînes, extrait) :**
- `run_commands.txt`

**Paramètres recommandés (un à la fois, puis small seeds) :**
- Utiliser plutôt preset/CLI ; si constants internes modifiées, documenter dans le rapport et snapshots.

### run_pipeline_A.py

**Rôle :** Orchestrateur : contrat run_id et pipeline bout en bout (générer → simuler → agréger → tracer → audit).
- Taille : ~**533** lignes
- Arguments CLI (extrait) : `--buildings_seed`, `--cross_enable`, `--cross_half_length_m`, `--cs_alpha`, `--cs_beta_delay_ms`, `--cs_cbr_cap`, `--cs_exp_scale_ms`, `--cs_gamma_cbr_col`, `--cs_gamma_cbr_delay`, `--cs_mac_efficiency`, `--cs_min_speed_mps`, `--cs_phy_overhead_us` …

**Dépendances principales (extrait) :**
`__future__.annotations`, `argparse`, `subprocess`, `sys`, `pathlib.Path`, `paths_A.ensure_run_dirs_a`, `paths_A.make_run_id`, `run_logging.log_command`, `run_logging.update_manifest`

**Structures clés :**
- Fonctions :
  - `_run(cmd, cwd)`
  - `main()`

**Paramètres recommandés (un à la fois, puis small seeds) :**
- Préférer les options CLI dans `run_pipeline_A.py` (`--scenarios/--rets/--n_seeds/--msg_rate_hz/--deadline_ms/--enable_congestion`).

### sim_v2x_A.py

**Rôle :** Simulation cœur : sorties paquet + champs de preuve.
- Taille : ~**642** lignes
- Arguments CLI (extrait) : `--attempt_spacing_ms`, `--buildings_path`, `--buildings_seed`, `--cs_alpha`, `--cs_beta_delay_ms`, `--cs_cbr_cap`, `--cs_exp_scale_ms`, `--cs_gamma_cbr_col`, `--cs_gamma_cbr_delay`, `--cs_mac_efficiency`, `--cs_min_speed_mps`, `--cs_phy_overhead_us` …

**Dépendances principales (extrait) :**
`__future__.annotations`, `argparse`, `dataclasses.dataclass`, `pathlib.Path`, `typing.Iterable`, `typing.Optional`, `numpy`, `pandas`, `progress_util.progress`, `paths_A.ensure_run_dirs_a`, `paths_A.make_run_id`, `paths_A.load_latest_run_id`, `paths_A.ensure_base_dirs_a`, `paths_A.get_base_paths_a`, `modules.mac_congestion.CongestionParams`, `modules.mac_congestion.compute_ncs_from_distances`, `modules.mac_congestion.compute_airtime_s`, `modules.mac_congestion.compute_cbr`, `modules.mac_congestion.p_collision_from_ncs`, `modules.mac_congestion.congestion_extra_delay_ms` …

**Structures clés :**
- Classes :
  - `Rect`
- Fonctions :
  - `clamp01(x)`
  - `load_traj(traj_path)`
  - `load_buildings(buildings_path)`
  - `_legacy_dirs()`
  - `_pick_run_id(arg_run_id)`
  - `parse_tx_ids(s, all_ids)`
  - `_tag_is_cross(tag, prefixes)`
  - `compute_delay_ms(distance_m, attempt_idx, attempt_spacing_ms, rng, impairment_b, extra_ms, exp_scale_ms)`
  - `simulate_one_seed(scenario, retrans, seed, msg_rate_hz, tx_ids_fixed, tx_mode, tx_k, tx_k_cross, tx_cross_prefixes, traj…)`
  - `main()`

**Artéfacts potentiels (scan de chaînes, extrait) :**
- `buildings__UrbMask__seed{seed_use}.csv`
- `results_packets__{args.scenario}__ret{args.retrans}__{seed_tag}.csv`
- `traj__Ref.csv`
- `traj__{args.scenario}.csv`
- `tunnel_config__Tunnel.csv`

**Paramètres recommandés (un à la fois, puis small seeds) :**
- Utiliser plutôt preset/CLI ; si constants internes modifiées, documenter dans le rapport et snapshots.

### 4.2 Modules (`03_sim_A/py/modules`)

### road_geometry.py

**Rôle :** Outils de géométrie routière.
- Taille : ~**194** lignes

**Dépendances principales (extrait) :**
`__future__.annotations`, `dataclasses.dataclass`, `typing.Union`, `numpy`

**Structures clés :**
- Classes :
  - `RefPlusGeometry`

**Paramètres recommandés (un à la fois, puis small seeds) :**
- Utiliser plutôt preset/CLI ; si constants internes modifiées, documenter dans le rapport et snapshots.

### mac_congestion.py

**Rôle :** Proxy congestion : n_cs/CBR/p_col et cong_delay_ms (queue).
- Taille : ~**166** lignes

**Dépendances principales (extrait) :**
`__future__.annotations`, `dataclasses.dataclass`, `typing.Optional`, `numpy`

**Structures clés :**
- Classes :
  - `CongestionParams`
- Fonctions :
  - `compute_airtime_s(pkt_bytes, phy_rate_mbps, mac_efficiency, phy_overhead_us)`
  - `compute_cbr(n_cs, msg_rate_hz, airtime_s, cbr_cap)`
  - `p_collision_from_ncs(n_cs, alpha_col, cbr, gamma_cbr_col)`
  - `congestion_extra_delay_ms(rng, n_cs, beta_delay_ms, exp_scale_ms, cbr, gamma_cbr_delay)`
  - `compute_ncs_from_distances(dist_all, tx_index, r_cs_m, active_mask, speed_all, min_speed_mps)`

**Paramètres recommandés (un à la fois, puis small seeds) :**
- Paramètres : mappings proxys (n_cs→CBR→p_col) et structure de queue cong_delay_ms ; valider Fig03/Fig04 après modifications.

### buildings_3d.py

**Rôle :** Géométrie bâtiments (support d_min).
- Taille : ~**163** lignes

**Dépendances principales (extrait) :**
`__future__.annotations`, `dataclasses.dataclass`, `pathlib.Path`, `typing.Iterable`, `numpy`, `pandas`

**Structures clés :**
- Classes :
  - `Rect2D`
  - `BuildingBlock` (bases : Rect2D)
- Fonctions :
  - `_pick_zone_by_x(x_mid, road_length_m)`
  - `generate_buildings()`
  - `buildings_to_dataframe(buildings)`
  - `save_buildings_csv(buildings, path)`
  - `load_buildings_csv(path)`
  - `as_rects(buildings)`

**Paramètres recommandés (un à la fois, puis small seeds) :**
- Utiliser plutôt preset/CLI ; si constants internes modifiées, documenter dans le rapport et snapshots.

### prop_city.py

**Rôle :** Dégradation urbaine : d_min→b, mélange LOS/NLOS, réflexion équivalente (option).
- Taille : ~**140** lignes

**Dépendances principales (extrait) :**
`__future__.annotations`, `typing.Iterable`, `typing.Protocol`, `typing.Tuple`, `numpy`

**Structures clés :**
- Classes :
  - `RectLike` (bases : Protocol)
- Fonctions :
  - `clamp01(x)`
  - `segment_intersects_rect(ax, ay, bx, by, rect)`
  - `segment_to_rect_min_distance(ax, ay, bx, by, rect)`
  - `blockage_strength_with_dmin(ax, ay, bx, by, buildings, transition_m)`
  - `p_success_los(distance_m)`
  - `p_success_nlos(distance_m)`
  - `refl_gain_db(d_min_m, b, gmax_db, d0_m)`

**Paramètres recommandés (un à la fois, puis small seeds) :**
- Paramètres : échelle d_min→b (T), courbes LOS/NLOS, coefficients de réflexion équivalente.

### prop_tunnel.py

**Rôle :** Dégradation tunnel : tunnel_u, bell+fade, injection de queue.
- Taille : ~**111** lignes

**Dépendances principales (extrait) :**
`__future__.annotations`, `dataclasses.dataclass`, `pathlib.Path`, `typing.Tuple`, `numpy`, `pandas`

**Structures clés :**
- Classes :
  - `TunnelConfig`
- Fonctions :
  - `clamp01(x)`
  - `tunnel_impairment_b(tx_x, rx_x, cfg)`

**Paramètres recommandés (un à la fois, puis small seeds) :**
- Paramètres : intervalle/transition, b_floor/b_peak/γ, échelle de queue.

### scenario_refplus.py

**Rôle :** Presets : paramètres par défaut figés.
- Taille : ~**65** lignes

**Dépendances principales (extrait) :**
`__future__.annotations`, `dataclasses.dataclass`, `dataclasses.asdict`, `dataclasses.field`

**Structures clés :**
- Classes :
  - `RefPlusGeometryConfig`
  - `TrafficIDMConfig`
  - `TrafficSignalConfig`
  - `RefPlusScenarioConfig`

**Paramètres recommandés (un à la fois, puis small seeds) :**
- Ajuster `scenario_*.py` ; re‑vérifier les tendances via Fig03/04/05/06/07.

### traffic_signals.py

**Rôle :** Dynamique trafic (IDM + feux).
- Taille : ~**54** lignes

**Dépendances principales (extrait) :**
`__future__.annotations`, `dataclasses.dataclass`

**Structures clés :**
- Classes :
  - `SignalPlan`

**Paramètres recommandés (un à la fois, puis small seeds) :**
- Utiliser plutôt preset/CLI ; si constants internes modifiées, documenter dans le rapport et snapshots.

### scenario_urbmaskplus.py

**Rôle :** Presets : paramètres par défaut figés.
- Taille : ~**43** lignes

**Dépendances principales (extrait) :**
`__future__.annotations`, `dataclasses.dataclass`, `dataclasses.asdict`, `dataclasses.field`

**Structures clés :**
- Classes :
  - `UrbMaskBuildingsConfig`
  - `UrbMaskPropagationConfig`
  - `UrbMaskScenarioConfig`

**Paramètres recommandés (un à la fois, puis small seeds) :**
- Ajuster `scenario_*.py` ; re‑vérifier les tendances via Fig03/04/05/06/07.

### traffic_idm.py

**Rôle :** Dynamique trafic (IDM + feux).
- Taille : ~**38** lignes

**Dépendances principales (extrait) :**
`__future__.annotations`, `dataclasses.dataclass`, `numpy`

**Structures clés :**
- Classes :
  - `IDMParams`
- Fonctions :
  - `idm_accel(v, dv, gap, p)`

**Paramètres recommandés (un à la fois, puis small seeds) :**
- Utiliser plutôt preset/CLI ; si constants internes modifiées, documenter dans le rapport et snapshots.

### scenario_tunnelplus.py

**Rôle :** Presets : paramètres par défaut figés.
- Taille : ~**17** lignes

**Dépendances principales (extrait) :**
`__future__.annotations`, `dataclasses.dataclass`, `dataclasses.asdict`, `dataclasses.field`, `prop_tunnel.TunnelConfig`

**Structures clés :**
- Classes :
  - `TunnelScenarioConfig`

**Paramètres recommandés (un à la fois, puis small seeds) :**
- Ajuster `scenario_*.py` ; re‑vérifier les tendances via Fig03/04/05/06/07.